# Visualizing OpenImpala Results with yt

OpenImpala uses the [AMReX](https://amrex-codes.github.io/amrex/) backend, which means its output plotfiles are natively supported by [yt](https://yt-project.org/) — a Python package designed for out-of-core analysis of large volumetric datasets.

This notebook demonstrates how to load and visualize OpenImpala results directly in Jupyter, without loading the entire dataset into RAM. This is critical for multi-billion voxel domains that would otherwise trigger Out-of-Memory errors.

**What you will learn:**
1. How to load an AMReX plotfile with `yt.load()`
2. How to create 2D slice plots of the solution field
3. How to create 1D profile plots (e.g., average flux along an axis)
4. How to extract numerical data for custom analysis

**Prerequisites:**
- An OpenImpala simulation run with `write_plotfile = 1`
- `yt` installed: `pip install yt`

## 0. Install dependencies

In [ ]:
!pip install -q yt matplotlib

## 1. Generate a plotfile with OpenImpala

To produce a plotfile, run OpenImpala with `write_plotfile = 1` in your inputs file:

```ini
# inputs file
filename = my_sample.tif
data_path = ./data/
results_path = ./results/
phase_id = 1
threshold_val = 0.5
direction = X
solver_type = FlexGMRES
write_plotfile = 1   # <-- This enables AMReX plotfile output
```

Or from the Python API:

```python
import openimpala as oi
import numpy as np

data = np.load("my_volume.npy").astype(np.int32)

with oi.Session():
    # The C++ solver writes plotfiles when write_plotfile=True
    img = oi._core.VoxelImage.from_numpy(data, max_grid_size=64)
    solver = oi._core.TortuosityHypre(
        img, 0.5, 0, oi.Direction.X, oi.SolverType.FlexGMRES,
        "./results/", 0.0, 1.0, 1, True,  # write_plotfile=True
    )
    tau = solver.value()
```

This creates an AMReX plotfile directory (e.g., `results/plt_tortuosity_X/`) containing the solution field and phase data.

## 2. Load the plotfile with yt

yt auto-detects AMReX plotfile format. Just point it at the directory.

In [ ]:
import yt

# Load the AMReX plotfile directory
# Replace with the path to your plotfile
PLOTFILE = "./results/plt_tortuosity_X"

ds = yt.load(PLOTFILE)

# Print available fields and domain info
print("Domain dimensions:", ds.domain_dimensions)
print("Domain left edge: ", ds.domain_left_edge)
print("Domain right edge:", ds.domain_right_edge)
print()
print("Available fields:")
for field in sorted(ds.field_list):
    print(f"  {field}")

Typical OpenImpala plotfiles contain:
- `('boxlib', 'phi')` — the computed potential/concentration field
- `('boxlib', 'phase')` — the thresholded phase map (0 = pore, 1 = solid)

## 3. 2D Slice Plots

`yt.SlicePlot` generates a 2D cross-section through the 3D domain. This is the most common visualization for transport fields — it shows the concentration gradient across the pore network.

**Key advantage:** yt reads data lazily from disk, so this works on datasets far larger than available RAM.

In [ ]:
# Slice through the centre of the domain, normal to the Y axis
slc = yt.SlicePlot(ds, "y", ("boxlib", "phi"))

# Customize the plot
slc.set_cmap(("boxlib", "phi"), "viridis")
slc.set_colorbar_label(("boxlib", "phi"), "Potential $\\phi$")
slc.annotate_title("Steady-State Concentration Field (Y-slice)")

slc.show()

In [ ]:
# Side-by-side: phase map and solution field
fig, axes = None, None  # yt manages its own figure

# Phase map — shows pore (0) vs solid (1)
slc_phase = yt.SlicePlot(ds, "y", ("boxlib", "phase"))
slc_phase.set_cmap(("boxlib", "phase"), "RdBu")
slc_phase.set_colorbar_label(("boxlib", "phase"), "Phase ID")
slc_phase.annotate_title("Phase Map")
slc_phase.show()

In [ ]:
# Slice in the XZ plane (normal to Y) at different depths
# Useful for inspecting how the solution varies through the sample
for z_frac in [0.25, 0.5, 0.75]:
    center = ds.domain_center.copy()
    center[2] = ds.domain_left_edge[2] + z_frac * (ds.domain_right_edge[2] - ds.domain_left_edge[2])

    slc = yt.SlicePlot(ds, "z", ("boxlib", "phi"), center=center)
    slc.set_cmap(("boxlib", "phi"), "viridis")
    slc.annotate_title(f"Z = {z_frac:.0%} of domain")
    slc.show()

## 4. 1D Profile Plots

`yt.ProfilePlot` computes bin-averaged quantities along a chosen axis. This is ideal for visualizing how the average potential (or flux) varies along the flow direction — the gradient of this curve is directly related to the effective diffusivity.

In [ ]:
# Average potential along the X axis (the flow direction)
# This should show a roughly linear gradient from phi=0 to phi=1
ad = ds.all_data()

profile = yt.ProfilePlot(
    ad,
    ("index", "x"),          # bin along X
    ("boxlib", "phi"),        # average this field
    weight_field=("index", "ones"),  # simple average (not volume-weighted)
)

profile.set_ylabel(("boxlib", "phi"), "Average Potential $\\langle\\phi\\rangle$")
profile.set_xlabel("Position along X")
profile.annotate_title("Through-Thickness Potential Profile")

profile.show()

**Interpreting the profile:**
- For a uniform medium with Dirichlet BCs ($\phi=0$ at inlet, $\phi=1$ at outlet), the profile is a straight line.
- For a heterogeneous porous medium, the profile deviates from linearity. Steeper gradients indicate regions of higher resistance (lower local diffusivity).
- The overall slope relates to the effective diffusivity: $D_{\text{eff}} = \text{flux} / |\nabla\phi|$.

## 5. Extracting Numerical Data

For custom analysis beyond built-in yt plots, extract data into NumPy arrays.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a 1D profile manually for more control
ad = ds.all_data()
prof = yt.create_profile(
    ad,
    ("index", "x"),
    ("boxlib", "phi"),
    n_bins=64,
    weight_field=("index", "ones"),
)

x = np.array(prof.x)
phi_avg = np.array(prof[("boxlib", "phi")])

# Plot with matplotlib for full customization
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.plot(x, phi_avg, 'o-', color='#1f77b4', ms=4, lw=1.5)
ax.set_xlabel("Position along flow direction (X)", fontweight='bold')
ax.set_ylabel("$\\langle\\phi\\rangle$", fontweight='bold', fontsize=14)
ax.set_title("Through-Thickness Potential Profile")
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Compute the effective gradient
dphi_dx = np.gradient(phi_avg, x)
print(f"Mean gradient dphi/dx = {np.mean(dphi_dx):.6f}")

In [ ]:
# Extract a 2D slice as a NumPy array (fixed resolution buffer)
slc = ds.slice("y", ds.domain_center[1])

# frb = Fixed Resolution Buffer — rasterizes the AMR data to a uniform grid
frb = slc.to_frb(
    width=ds.domain_width[0],
    resolution=(512, 512),  # output pixel resolution
)

phi_2d = np.array(frb[("boxlib", "phi")])

fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
im = ax.imshow(phi_2d, origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax, label='Potential $\\phi$')
ax.set_title("Solution Field (Y-midplane)")
ax.set_xlabel("X")
ax.set_ylabel("Z")
plt.tight_layout()
plt.show()

print(f"Extracted array shape: {phi_2d.shape}")
print(f"Value range: [{phi_2d.min():.4f}, {phi_2d.max():.4f}]")

## 6. Working with Large Datasets

The key advantage of yt is **lazy loading** — it only reads the data needed for the current operation. This means the techniques above work identically on a $100^3$ test domain and a $2000^3$ synchrotron dataset.

### Tips for large datasets

| Technique | Memory Usage | When to Use |
|-----------|-------------|-------------|
| `yt.SlicePlot` | Low — reads one 2D plane | Quick visual inspection |
| `yt.ProjectionPlot` | Medium — integrates along an axis | Overview of field structure |
| `yt.ProfilePlot` | Low — bin-averages on the fly | Quantitative 1D analysis |
| `ds.all_data()` | **High** — reads everything | Only for small datasets or HPC |
| `ds.region(...)` | Configurable | Sub-volume analysis |

### Selecting sub-regions

```python
# Analyse only a sub-volume (avoids loading the full dataset)
left = ds.domain_left_edge + 0.25 * ds.domain_width
right = ds.domain_left_edge + 0.75 * ds.domain_width
region = ds.region(center=0.5*(left+right), left_edge=left, right_edge=right)

# All yt operations work on regions too
profile = yt.ProfilePlot(region, ("index", "x"), ("boxlib", "phi"))
```

### Remote visualization via ParaView

For fully interactive 3D rendering of large datasets, consider using ParaView's client-server mode. See the [ParaView visualization guide](../docs/user-guide/visualization-paraview.md) for instructions on connecting a local ParaView GUI to a remote HPC server.

## References

- [yt documentation](https://yt-project.org/docs/dev/)
- [yt AMReX frontend](https://yt-project.org/docs/dev/examining/loading_data.html#amrex-boxlib-data)
- [AMReX plotfile format](https://amrex-codes.github.io/amrex/docs_html/Visualization.html)
- [OpenImpala tutorials](../tutorials/)